<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Download_raw_data_for_webpages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading raw analysis data and input/output files
For each JARVIS-ID, there is a respective XML document which can be downloaded and all the raw data can be parsed from that XML. Also, the raw input/output
files for each calculation is distributed through Figshare repository
, links for each are given at the bottom of the webpage.



In [ ]:
!pip install -U jarvis-tools

In [ ]:
import jarvis
jarvis.__version__

In [ ]:
from jarvis.db.webpages import Webpage
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
w = Webpage(jid='JVASP-1002')
keys = w.list_keys()

In [ ]:
# Get and plot electronic DOS from OptB88vdW after  SCF
edos_data = w.get_dft_electron_dos()
ens = np.array(edos_data['edos_energies'].strip("'").split(','),dtype='float')
tot_dos_up = np.array(edos_data['total_edos_up'].strip("'").split(','),dtype='float') 
plt.plot(ens,tot_dos_up)
plt.xlabel('Energy (eV')
plt.ylabel('DOS (arb.unit.)')

In [ ]:
mbj_dl= w.get_dft_mbj_dielectric_function()

In [ ]:

plt.plot(mbj_dl['energies'],mbj_dl['real_xx'])
plt.xlabel('Energy (eV')
plt.ylabel('Real part Eps-x')

In [ ]:
from jarvis.db.figshare import data,get_jid_data
import os,tempfile,io,zipfile
x = get_jid_data(jid='JVASP-1002',dataset='dft_3d')['raw_files']
print(x)

In [ ]:
import requests
for i in x:
  if 'DFT-SCF' in i:
    print (i)
    r = requests.get(i.split(',')[2]).content
    
    

In [ ]:
z = zipfile.ZipFile(io.BytesIO(r))

In [ ]:
z.namelist()

In [ ]:
from jarvis.io.vasp.inputs import Poscar
from jarvis.io.vasp.outputs import Vasprun
import tempfile

initial = Poscar.from_string(z.read('POSCAR').decode('utf-8'))
final = Poscar.from_string(z.read('POSCAR').decode('utf-8'))
fd, path = tempfile.mkstemp()
with os.fdopen(fd, "w") as tmp:
    tmp.write(z.read('vasprun.xml').decode('utf-8'))
vrun = Vasprun(path)

In [ ]:
vrun.all_energies

In [ ]:
vrun.all_forces

In [ ]:
vrun.get_indir_gap